# How much does raw-MSE planning detour in band 1?

This read-only audit opens the saved **band 1** rollouts from the narrow square-band half-map experiment. It uses the real ball positions after every simulator action. No GPU is needed.

The two goals are chosen by vertical position. The upper goal has the lowest y, and the lower goal has the highest y. We summarize every rollout for those goals and show exactly three paths per goal when available.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import io
import json
import numpy as np
import pandas as pd
import torch

drive.mount('/content/drive')
RUNS=Path('/content/drive/MyDrive/datasets/curriculum_cost_adapter/experience_temporal_half_square_v1')
RUN_ID=None  # Optional. Set to the ID printed by the collection notebook.
SUCCESS_RADIUS=4.5
MOVE_TOLERANCE=1e-5

## Load the real simulator trajectories

The default is the latest run whose band 1 finished collecting under raw latent MSE. The cell prints the run ID for verification. A saved band 1 adapter is unnecessary.

In [ ]:
def read_json(path):
    return json.loads(path.read_text())

def find_run(root, run_id):
    candidates=[root/run_id] if run_id else sorted(root.iterdir(),reverse=True)
    for candidate in candidates:
        status=candidate/'stages/band_1.json'
        if not status.is_file():
            continue
        stage=read_json(status)
        if stage['status']=='complete' and stage.get('condition',{}).get('collector')=='raw':
            return candidate
    raise ValueError('No completed raw-MSE band 1 found. Check RUNS and RUN_ID.')

def load_paths(run):
    stage=read_json(run/'stages/band_1.json')
    manifest=read_json(run/'experiment.json')['manifest']
    cases=manifest['stages'][0]
    expected=hashlib.sha256(json.dumps(cases,sort_keys=True,allow_nan=False).encode()).hexdigest()
    if stage['cases_digest']!=expected or stage['episodes']!=len(cases):
        raise ValueError('Band 1 cases differ from the saved manifest.')
    goals=sorted({tuple(case['goal']) for case in cases},key=lambda goal:(goal[1],goal[0]))
    if len(goals)<2 or goals[0][1]==goals[-1][1]:
        raise ValueError('Band 1 must contain upper and lower goals.')
    result={}
    for label,goal in [('Upper goal',goals[0]),('Lower goal',goals[-1])]:
        rows=[]
        for case in cases:
            if tuple(case['goal'])!=goal:
                continue
            case_id=case['case_id']
            record=read_json(run/f'episodes/band_1/{case_id}.json')
            if (record['case']!=case or record['runtime_id']!=stage['runtime_id']
                    or record['condition']!=stage['condition']):
                raise ValueError(f'Episode provenance differs for {case_id}.')
            raw_path=(run/record['raw_path']).resolve()
            raw_path.relative_to(run.resolve())
            payload=raw_path.read_bytes()
            if hashlib.sha256(payload).hexdigest()!=record['raw_sha256']:
                raise ValueError(f'Raw observations changed for {case_id}.')
            raw=torch.load(io.BytesIO(payload),map_location='cpu',weights_only=True)
            xy=raw['states'][:,:2].numpy().astype(float)
            steps=record['micro_steps']
            if (raw['case']!=case or raw['runtime_id']!=stage['runtime_id']
                    or raw['success']!=record['success'] or len(xy)!=steps+1
                    or not np.array_equal(raw['primitive_steps'].numpy(),np.arange(steps+1))
                    or not np.isfinite(xy).all()):
                raise ValueError(f'Raw observations differ from the record for {case_id}.')
            rows.append(dict(case=case_id,goal=goal,success=record['success'],xy=xy))
        result[label]=rows
    return result

run=find_run(RUNS,RUN_ID)
paths=load_paths(run)
print('Run ID:',run.name)
print('Collector:',read_json(run/'stages/band_1.json')['condition']['collector'])
for label,rows in paths.items():
    print(label,rows[0]['goal'],'·',len(rows),'rollouts')

## Quantify detours

**Extra travel** is the distance traveled minus the straight distance from the start to the actual endpoint. This accounts for the planner stopping within 4.5 units of the goal. **Away distance** adds up the increases in goal distance on individual actions. **Worst backtrack** is the largest increase above the closest distance reached so far. These are physical measurements in the right room.

In [ ]:
def measure_path(row):
    xy=row['xy']
    distance=np.linalg.norm(xy-row['goal'],axis=1)
    change=np.diff(distance)
    length=np.linalg.norm(np.diff(xy,axis=0),axis=1).sum()
    displacement=np.linalg.norm(xy[-1]-xy[0])
    return dict(case=row['case'],success=row['success'],actions=len(change),
                initial_distance=distance[0],final_distance=distance[-1],path_length=length,
                extra_travel=max(0.,length-displacement),away_steps=int((change>MOVE_TOLERANCE).sum()),
                away_distance=np.maximum(change,0).sum(),
                worst_backtrack=(distance-np.minimum.accumulate(distance)).max())

def choose_three(rows, stats):
    # Use successful rollouts for the visual comparison when at least three exist.
    by_case={r['case']:r for r in rows}
    pool=[s for s in stats if s['success']]
    if len(pool)<3:
        pool=stats
    ordered=sorted(pool,key=lambda s:(s['extra_travel'],s['case']))
    indices=np.linspace(0,len(ordered)-1,min(3,len(ordered)),dtype=int)
    return [by_case[ordered[i]['case']] for i in indices]

shown={}
tables=[]
for label,rows in paths.items():
    stats=[measure_path(r) for r in rows]
    chosen=choose_three(rows,stats)
    shown[label]=chosen
    ids={r['case'] for r in chosen}
    tables.extend(dict(group=label,goal=r['goal'],plotted=r['case'] in ids,**s)
                  for r,s in zip(rows,stats))

table=pd.DataFrame(tables)
print('All rollouts for the two goals')
display(table.round(2))
print('Goal summaries')
summary=table.groupby('group',sort=False).agg(rollouts=('case','size'),successes=('success','sum'),
    median_extra_travel=('extra_travel','median'),median_away_steps=('away_steps','median'),
    median_away_distance=('away_distance','median'),max_backtrack=('worst_backtrack','max'))
display(summary.round(2))
table.to_csv('/content/raw_mse_band1_detours.csv',index=False)

## Three trajectories per goal

The paths span the least, middle, and most extra travel among successful episodes. If fewer than three succeeded, the plots also consider failed episodes. The lower panels show the same trajectories as goal distance versus action. An upward segment means the ball moved away from the goal.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

fig,axes=plt.subplots(2,2,figsize=(13,9),constrained_layout=True)
for column,(label,rows) in enumerate(shown.items()):
    spatial,progress=axes[:,column]
    goal=rows[0]['goal']
    spatial.add_patch(Circle(goal,SUCCESS_RADIUS,fill=False,edgecolor='gray',ls=':'))
    spatial.scatter(*goal,marker='*',c='black',s=120,zorder=5)
    for color,row in zip(('tab:blue','tab:orange','tab:green'),rows):
        xy=row['xy']
        stats=measure_path(row)
        tag=f"{row['case']} · {stats['extra_travel']:.1f} extra"
        if not row['success']:
            tag+=' · failed'
        spatial.plot(*xy.T,color=color,marker='.',markersize=3,label=tag)
        spatial.scatter(*xy[0],s=50,facecolors='none',edgecolors=color,zorder=4)
        spatial.scatter(*xy[-1],s=38,marker='x',color=color,zorder=4)
        progress.plot(np.arange(len(xy)),np.linalg.norm(xy-goal,axis=1),color=color,marker='.',markersize=3)
    points=np.vstack([*(r['xy'] for r in rows),np.asarray(goal)-SUCCESS_RADIUS,
                      np.asarray(goal)+SUCCESS_RADIUS])
    lo,hi=points.min(axis=0)-1,points.max(axis=0)+1
    spatial.set(xlim=(lo[0],hi[0]),ylim=(hi[1],lo[1]),aspect='equal',xlabel='Ball x',ylabel='Ball y',
                title=f'{label} {goal}')
    spatial.legend(loc='best',fontsize=8)
    progress.axhline(SUCCESS_RADIUS,ls=':',color='gray')
    progress.set(xlabel='Executed action',ylabel='Distance to goal (units)')
    progress.grid(alpha=.18)
fig.suptitle('Band 1 raw latent MSE · open circle = start · × = endpoint · star = goal')
fig.savefig('/content/raw_mse_band1_detours.png',dpi=150,bbox_inches='tight')
plt.show()